In [16]:
# buckets for activation values

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from Data.MNIST_Loader import load_and_preprocess_data

# Load model
model = tf.keras.models.load_model("Models/best_model.h5")

# Load a batch of inputs
ds_train, ds_val, ds_test = load_and_preprocess_data()
for batch in ds_val.take(1):
    inputs, _ = batch  # shape: (batch_size, 28, 28, 1)

# Identify the flatten layer and create an inference function for it
flatten_layer = None
for layer in model.layers:
    if 'flatten' in layer.name.lower():
        flatten_layer = layer
        break

if flatten_layer is None:
    raise ValueError("No Flatten layer found in the model.")

# Use the functional API to create a new model from the original model's input tensors to the flatten layer output
flatten_model = Model(inputs=model.inputs, outputs=flatten_layer.output)

# Run inference to get flattened activations
a = flatten_model.predict(inputs)

# Identify the first Dense layer to get weights and biases
first_dense_layer = None
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        first_dense_layer = layer
        break

if first_dense_layer is None:
    raise ValueError("No Dense layer found in the model.")

kernel, bias = first_dense_layer.get_weights()

# Compute pre-activation z = wa + b
z = np.dot(a, kernel) + bias

# Flatten z and bucket as desired
z_flat = z.flatten()

bucket_edges = np.concatenate(([-np.inf, -1], np.arange(-1, 1, 0.1), [1, np.inf]))
bucket_labels = ['Less than -1'] + \
                [f"{round(left,1)} to {round(left+0.1,1)}" for left in np.arange(-1,1,0.1)] + \
                ['Greater than 1']

bucket_indices = np.digitize(z_flat, bucket_edges) - 1
counts = {label: np.sum(bucket_indices == i) for i, label in enumerate(bucket_labels)}

for label in bucket_labels:
    print(f"{label}: {counts[label]}")


2025-10-12 19:54:43.806117: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-10-12 19:54:43.806281: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Less than -1: 11601
-1.0 to -0.9: 0
-0.9 to -0.8: 1267
-0.8 to -0.7: 1385
-0.7 to -0.6: 1537
-0.6 to -0.5: 1601
-0.5 to -0.4: 1790
-0.4 to -0.3: 2006
-0.3 to -0.2: 2067
-0.2 to -0.1: 2340
-0.1 to -0.0: 2463
-0.0 to 0.1: 2699
0.1 to 0.2: 2674
0.2 to 0.3: 2536
0.3 to 0.4: 2409
0.4 to 0.5: 2411
0.5 to 0.6: 2302
0.6 to 0.7: 2071
0.7 to 0.8: 2009
0.8 to 0.9: 1882
0.9 to 1.0: 1805
Greater than 1: 1645
